# Практика 13. Построение воспроизводимого аналитического pipeline

Это итоговая пара курса. Всё, что мы делали по отдельности — загрузку (практика 7),
очистку (8), агрегацию (9), статистику (10 и 12), графики (11) — сегодня соберём
в **один запускаемый проект**.

**Задача пары:** сделать так, чтобы весь анализ выполнялся одной командой

```
python run.py
```

и давал одинаковый результат у вас, у преподавателя и у вас же через полгода.

| Часть | О чём |
|---|---|
| 1 | Почему ноутбук — плохое место для готового анализа |
| 2 | Структура проекта и слои обработки |
| 3–6 | Модули: загрузка, подготовка, расчёт, визуализация |
| 7 | Параметризация: конфиг вместо правок в коде |
| 8 | Сборка и запуск pipeline |
| 9 | Проверка воспроизводимости |
| 10 | Артефакты и манифест запуска |

> **Главная мысль пары:** анализ, который нельзя повторить, — это не анализ, а разовый
> фокус. Воспроизводимость означает, что другой человек с теми же исходными данными
> получит ровно те же числа, не задавая вам ни одного вопроса.

---
# Часть 1. Чем плох ноутбук

Ноутбук — отличный инструмент для **исследования**: что-то попробовал, посмотрел,
поправил. Но как способ хранить готовый анализ он разваливается. Пять симптомов,
которые вы наверняка уже видели на этом курсе:

| Симптом | Чем кончается |
|---|---|
| ячейки запускаются в произвольном порядке | результат зависит от того, что вы нажимали; у соседа другие числа |
| переменная из ячейки 7 используется в ячейке 40 | удалили ячейку 7 — всё сломалось, и непонятно почему |
| один и тот же код скопирован в пяти местах | нашли ошибку — надо исправить в пяти местах, исправили в трёх |
| параметры (даты, пороги) разбросаны по коду | чтобы пересчитать за другой месяц, надо найти и поменять их все |
| результат живёт только внутри ноутбука | коллеге нечего отправить, кроме файла на 60 ячеек |

Главный тест: **откройте «Restart Kernel and Run All» на своём ноутбуке с практики 9.
Он отработает без ошибок?** Если нет — ваш анализ невоспроизводим.

Решение — вынести код в модули, параметры в конфиг, а ноутбук оставить для того,
для чего он хорош: для черновика и для демонстрации результата.

---
# Часть 2. Структура проекта и слои

Соберём такую структуру:

```
trips-pipeline/
├── config.py              параметры: пути, период, пороги
├── run.py                 точка входа: запускает весь pipeline
├── src/
│   ├── load.py            слой 1: прочитать сырые данные
│   ├── prepare.py         слой 2: очистить и привести к единому виду
│   ├── analyze.py         слой 3: посчитать метрики
│   └── visualize.py       слой 4: построить графики
├── data/
│   ├── raw/               исходные файлы — НИКОГДА не меняем
│   └── processed/         результат очистки и расчётов
└── reports/
    ├── figures/           графики
    └── report.md          итоговый отчёт
```

**Четыре слоя и правило для каждого:**

| Слой | Что делает | Чего НЕ делает |
|---|---|---|
| `load` | читает файлы как есть, минимум преобразований | не чистит и не считает |
| `prepare` | типы, дубликаты, нормализация, аномалии | не считает метрики |
| `analyze` | агрегаты, KPI, проверка гипотез | не читает файлы и не рисует |
| `visualize` | графики по готовым метрикам | не считает метрики заново |

Зачем так строго: если каждый слой делает своё, любой из них можно заменить, не трогая
остальные. Пришли данные в другом формате — меняется только `load`. Попросили другой
график — только `visualize`.

**Главное правило работы с данными:** папка `data/raw/` **только для чтения**.
Что бы ни случилось, исходные данные остаются нетронутыми — иначе откатиться будет некуда.

In [2]:
# ЯЧЕЙКА-ГЕНЕРАТОР ИСХОДНЫХ ДАННЫХ. Запустите её один раз.
# Создаёт data/raw/ — две месячные выгрузки "как из системы" и справочник перевозчиков.
# Данные намеренно сырые: их придётся чистить внутри pipeline.

import csv
import random
from pathlib import Path

random.seed(13)

routes = {
    "Москва-Казань": 820, "Москва-Санкт-Петербург": 710, "Москва-Нижний Новгород": 420,
    "Москва-Воронеж": 520, "Казань-Самара": 360, "Екатеринбург-Челябинск": 220,
    "Новосибирск-Омск": 680, "Санкт-Петербург-Петрозаводск": 430,
}
carrier_variants = {
    "ТрансЛогистик": ["ТрансЛогистик", "транслогистик", 'ООО "ТрансЛогистик"'],
    "РусКарго": ["РусКарго", "рускарго", "  РусКарго "],
    "СибПуть": ["СибПуть", "сибпуть"],
    "ВолгаТранс": ["ВолгаТранс", "волгатранс "],
    "АвтоДорСервис": ["АвтоДорСервис", "автодорсервис"],
}
tariffs = {"Контейнер": 7.5, "Нефтепродукты": 9.2, "Зерно": 5.1,
           "Металл": 6.8, "Лес": 4.6, "Стройматериалы": 5.9}

Path("data/raw").mkdir(parents=True, exist_ok=True)


def make_month(month, n, start_id, days):
    rows = []
    for i in range(start_id, start_id + n):
        route, distance = random.choice(list(routes.items()))
        cargo = random.choice(list(tariffs))
        canon = random.choice(list(carrier_variants))
        weight = round(min(22.0, max(2.0, random.gauss(13, 4.5))), 1)
        delay = random.choice([1, 2, 3, 4, 6, 8]) if random.random() < 0.28 else 0
        cost = distance * weight * tariffs[cargo] * random.uniform(0.94, 1.09)

        row = {
            "trip_id": f"T{i:04d}",
            "date": f"{random.randint(1, days):02d}.{month:02d}.2025",
            "route": route,
            "carrier": random.choice(carrier_variants[canon]),
            "cargo_type": cargo,
            "weight_t": str(weight).replace(".", ","),
            "distance_km": distance,
            "cost_rub": f"{cost:.2f}".replace(".", ","),
            "delay_h": "" if random.random() < 0.05 else delay,
        }
        # немного брака: потерянный разряд в стоимости и нулевой вес
        r = random.random()
        if r < 0.03:
            row["cost_rub"] = f"{cost / 10:.2f}".replace(".", ",")
        elif r < 0.05:
            row["weight_t"] = "0"
        rows.append(row)
    # пара дублей в каждой выгрузке
    rows += [dict(rows[5]), dict(rows[40])]
    return rows


for month, n, start, days in [(4, 240, 1, 30), (5, 260, 1001, 31)]:
    rows = make_month(month, n, start, days)
    path = f"data/raw/trips_2025_{month:02d}.csv"
    with open(path, "w", encoding="cp1251", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()), delimiter=";")
        w.writeheader()
        w.writerows(rows)
    print(f"  {path} — {len(rows)} строк (cp1251, разделитель ';')")

with open("data/raw/carriers.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["carrier", "city", "rating"])
    w.writeheader()
    for name, city, rating in [("ТрансЛогистик", "Москва", 4.6), ("РусКарго", "Москва", 4.1),
                               ("СибПуть", "Новосибирск", 3.8), ("ВолгаТранс", "Казань", 4.4),
                               ("АвтоДорСервис", "Воронеж", 3.5)]:
        w.writerow({"carrier": name, "city": city, "rating": rating})
print("  data/raw/carriers.csv — справочник перевозчиков")
print()
print("Готово. Сырые данные лежат в data/raw/ и до конца пары остаются неизменными.")

  data/raw/trips_2025_04.csv — 242 строк (cp1251, разделитель ';')
  data/raw/trips_2025_05.csv — 262 строк (cp1251, разделитель ';')
  data/raw/carriers.csv — справочник перевозчиков

Готово. Сырые данные лежат в data/raw/ и до конца пары остаются неизменными.


In [3]:
from pathlib import Path

for folder in ["src", "data/processed", "reports/figures"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

# чтобы папка src стала пакетом и из неё можно было импортировать
Path("src/__init__.py").write_text("", encoding="utf-8")

print("структура проекта готова:")
for p in sorted(Path(".").glob("**/*")):
    if ".ipynb_checkpoints" not in str(p) and "__pycache__" not in str(p):
        print("  ", p)

структура проекта готова:
   config.py
   data
   data\processed
   data\processed\kpi_by_carrier.csv
   data\processed\kpi_by_month.csv
   data\processed\run_manifest.json
   data\processed\trips_clean.csv
   data\raw
   data\raw\carriers.csv
   data\raw\trips_2025_04.csv
   data\raw\trips_2025_05.csv
   data\trips_stat.csv
   LW1 (1).ipynb
   LW2 (1).ipynb
   reports
   reports\figures
   reports\figures\otd_by_carrier.png
   reports\figures\revenue_by_month.png
   reports\report.md
   requirements.txt
   run.py
   src
   src\__init__.py
   src\analyze.py
   src\load.py
   src\prepare.py
   src\visualize.py


---
# Часть 3. Слой 1: загрузка

Первый модуль читает сырые файлы и больше ничего не делает. Обратите внимание
на две вещи: функция принимает **путь параметром** (а не прячет его внутри)
и возвращает данные, ничего не печатая кроме информационного сообщения.

In [4]:
load_code = """
# -*- coding: utf-8 -*-
'''Слой 1: загрузка сырых данных. Читает файлы как есть, ничего не исправляет.'''
from pathlib import Path

import pandas as pd


def load_trips(raw_dir, pattern="trips_*.csv"):
    '''Читает все месячные выгрузки из raw_dir и склеивает в одну таблицу.'''
    files = sorted(Path(raw_dir).glob(pattern))
    if not files:
        raise FileNotFoundError(f"в {raw_dir} нет файлов по шаблону {pattern}")

    frames = []
    for path in files:
        part = pd.read_csv(path, sep=";", encoding="cp1251")
        part["source_file"] = path.name          # помним, откуда пришла строка
        frames.append(part)
        print(f"    {path.name}: {len(part)} строк")

    return pd.concat(frames, ignore_index=True)


def load_carriers(raw_dir, filename="carriers.csv"):
    '''Читает справочник перевозчиков.'''
    return pd.read_csv(Path(raw_dir) / filename)
"""

Path("src/load.py").write_text(load_code.lstrip(), encoding="utf-8")
print("создан src/load.py")

создан src/load.py


In [5]:
import sys

# чтобы Python видел пакет src из текущей папки
if "" not in sys.path:
    sys.path.insert(0, "")

from src.load import load_trips, load_carriers

raw = load_trips("data/raw")
carriers = load_carriers("data/raw")

print()
print("загружено строк:", len(raw))
print("перевозчиков в справочнике:", len(carriers))
raw.head(3)

    trips_2025_04.csv: 242 строк
    trips_2025_05.csv: 262 строк

загружено строк: 504
перевозчиков в справочнике: 5


,trip_id,date,route,carrier,cargo_type,weight_t,distance_km,cost_rub,delay_h,source_file
0,T0001,06.04.2025,Казань-Самара,РусКарго,Зерно,"9,2",360,"17501,71",0.0,trips_2025_04.csv
1,T0002,01.04.2025,Казань-Самара,волгатранс,Контейнер,"7,6",360,"21161,46",8.0,trips_2025_04.csv
2,T0003,25.04.2025,Казань-Самара,волгатранс,Металл,"20,6",360,"51085,27",2.0,trips_2025_04.csv


### Задание 1

1. Запустите `load_trips` с шаблоном `"trips_2025_05*.csv"` — загрузится только май.
   Сколько строк?
2. Что произойдёт, если передать несуществующую папку? Проверьте.
   Почему лучше, чтобы функция падала с понятным сообщением, а не возвращала
   пустую таблицу?
3. Добавьте в `load.py` функцию `load_all(raw_dir)`, которая возвращает **словарь**
   с двумя таблицами: `{"trips": ..., "carriers": ...}`. Так точка входа будет
   получать все данные одним вызовом.

In [6]:
# 1)
from src.load import load_trips
raw_dir = "data/raw"
trips_may = load_trips(raw_dir, pattern = 'trips_2025_05*.csv')

print('Количество строк:', len(trips_may))
display(trips_may.head())

# 2)
#load_trips('data/non_papka', pattern = 'trips_2025_05*.csv')

    trips_2025_05.csv: 262 строк
Количество строк: 262


,trip_id,date,route,carrier,cargo_type,weight_t,distance_km,cost_rub,delay_h,source_file
0,T1001,11.05.2025,Москва-Казань,СибПуть,Зерно,"14,6",820,"58730,14",0.0,trips_2025_05.csv
1,T1002,15.05.2025,Москва-Воронеж,АвтоДорСервис,Стройматериалы,"12,7",520,"39259,60",0.0,trips_2025_05.csv
2,T1003,27.05.2025,Москва-Казань,автодорсервис,Нефтепродукты,"13,2",820,"99389,80",6.0,trips_2025_05.csv
3,T1004,03.05.2025,Москва-Санкт-Петербург,сибпуть,Металл,"15,3",710,"75291,87",0.0,trips_2025_05.csv
4,T1005,10.05.2025,Казань-Самара,РусКарго,Лес,"14,3",360,"25556,58",0.0,trips_2025_05.csv


In [7]:
# 3)
load_all_code ='''
def load_all(raw_dir, pattern="trips_*.csv"):
    return {"trips": load_trips(raw_dir, pattern),
            "carriers": load_carriers(raw_dir)}'''
with open("src/load.py", "a", encoding="utf-8") as f:
    f.write(load_all_code)

2. При передаче несуществующей папки функция падает в ошибку. Это лучше, тк пустая таблица может создать впечатление, что данные просто отсутствуют, а также это помагает быстрее обнаружить проблему.

---
# Часть 4. Слой 2: подготовка

Этот модуль повторяет то, что мы делали на практике 8, но оформленное функциями.
Каждый шаг очистки — отдельная функция, и все они собираются в `prepare_trips`.

Важная деталь: функция возвращает **две** вещи — очищенные данные и отчёт о том,
что было сделано. Отчёт понадобится, чтобы заметить, если в следующем месяце
данные придут другими.

In [8]:
prepare_code = """
# -*- coding: utf-8 -*-
'''Слой 2: приведение сырых данных к пригодному для анализа виду.'''
import pandas as pd

CANON_CARRIERS = {
    "транслогистик": "ТрансЛогистик",
    "рускарго": "РусКарго",
    "сибпуть": "СибПуть",
    "волгатранс": "ВолгаТранс",
    "автодорсервис": "АвтоДорСервис",
}
TARIFFS = {"Контейнер": 7.5, "Нефтепродукты": 9.2, "Зерно": 5.1,
           "Металл": 6.8, "Лес": 4.6, "Стройматериалы": 5.9}


def to_canon(value, canon_map=CANON_CARRIERS):
    '''Приводит название перевозчика к каноническому виду.'''
    s = str(value).strip().lower()
    for prefix in ('ооо "', "ооо ", 'ао "', "ао "):
        if s.startswith(prefix):
            s = s[len(prefix):]
    s = s.strip('"').strip()
    return canon_map.get(s, value)


def to_number(series):
    '''Превращает текст с десятичной запятой в число.'''
    return pd.to_numeric(
        series.astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def prepare_trips(raw, max_cost_ratio=5.0):
    '''Очищает выгрузку. Возвращает (clean_df, report_dict).'''
    report = {"строк получено": len(raw)}
    df = raw.copy()

    df = df.drop_duplicates(subset=["trip_id"], keep="first")
    report["удалено дублей"] = report["строк получено"] - len(df)

    df["cost_rub"] = to_number(df["cost_rub"])
    df["weight_t"] = to_number(df["weight_t"])
    df["delay_h"] = pd.to_numeric(df["delay_h"], errors="coerce").fillna(0)
    df["date"] = pd.to_datetime(df["date"], format="%d.%m.%Y", errors="coerce")
    report["дат не распознано"] = int(df["date"].isna().sum())

    df["carrier"] = df["carrier"].apply(to_canon)
    df["cargo_type"] = df["cargo_type"].str.strip().str.capitalize()
    report["перевозчиков после нормализации"] = int(df["carrier"].nunique())

    bad_weight = df["weight_t"] <= 0
    report["удалено: нулевой вес"] = int(bad_weight.sum())
    df = df[~bad_weight]

    expected = df["weight_t"] * df["distance_km"] * df["cargo_type"].map(TARIFFS)
    bad_cost = (expected / df["cost_rub"]) > max_cost_ratio
    report["удалено: стоимость не бьётся с тарифом"] = int(bad_cost.sum())
    df = df[~bad_cost]

    df["month"] = df["date"].dt.to_period("M").astype(str)
    df["is_delayed"] = df["delay_h"] > 0
    df["cost_per_km"] = (df["cost_rub"] / df["distance_km"]).round(2)

    report["строк осталось"] = len(df)
    report["доля потерь, %"] = round((1 - len(df) / report["строк получено"]) * 100, 1)
    return df.reset_index(drop=True), report
"""

Path("src/prepare.py").write_text(prepare_code.lstrip(), encoding="utf-8")
print("создан src/prepare.py")

создан src/prepare.py


In [9]:
from src.prepare import prepare_trips

clean, quality = prepare_trips(raw)

print("ОТЧЁТ О КАЧЕСТВЕ")
for k, v in quality.items():
    print(f"  {k:<38} {v}")
print()
print("перевозчики:", sorted(clean["carrier"].unique()))

ОТЧЁТ О КАЧЕСТВЕ
  строк получено                         504
  удалено дублей                         4
  дат не распознано                      0
  перевозчиков после нормализации        5
  удалено: нулевой вес                   10
  удалено: стоимость не бьётся с тарифом 13
  строк осталось                         477
  доля потерь, %                         5.4

перевозчики: ['АвтоДорСервис', 'ВолгаТранс', 'РусКарго', 'СибПуть', 'ТрансЛогистик']


### Задание 2

1. Запустите `prepare_trips` с разными значениями `max_cost_ratio`: 1.05, 2.0, 5.0,
   9.0 и 11.0. Постройте таблицу «порог → сколько строк удалено».
   При каких порогах результат одинаковый? Почему? Что это говорит о том,
   насколько аккуратно нужно подбирать этот параметр?
2. В `prepare.py` нет обработки пропусков в `weight_t`. Добавьте в отчёт строку
   «пропусков в весе» (считать, но не заполнять).
3. Функция `to_canon` возвращает исходное значение, если не нашла соответствия.
   Допишите в `prepare_trips` проверку: если после нормализации перевозчиков
   не ровно 5, вывести предупреждение со списком незнакомых названий.

> Пункт 3 — важный приём. Pipeline запускается каждый месяц без вашего присмотра,
> и он должен **сам сообщать**, когда в данных появилось что-то новое.

In [10]:
# 1) таблица «порог -> удалено»
import pandas as pd
from src.prepare import prepare_trips
from src.load import load_trips

raw = load_trips("data/raw")

print(f"{'порог':>4}  {'удалено по стоимости':>15}")
for threshold in (1.05, 2.0, 5.0, 9.0, 11.0):
    processed, report = prepare_trips(raw, max_cost_ratio=threshold)
    print(f"{threshold:>4}  {report['удалено: стоимость не бьётся с тарифом']:>10}")

    trips_2025_04.csv: 242 строк
    trips_2025_05.csv: 262 строк
порог  удалено по стоимости
1.05          66
 2.0          13
 5.0          13
 9.0          13
11.0           0


1) 
> При порогах 2.0, 5.0, 9.0 удаляется одинаковое количество строк - 13.
> Так как между этими значениями нет дополнительных записей, которые подходят под условие удаления, поэтому изменение не влият на результат.
> Порог нужно подбирать аккуратно, тк слишком маленькое значение приводит к удалению большого количества строк, в том числе возможно корректных. Слишком большое значение может пропустить ошибочное значение.

In [13]:
# 2)-3)
weight_t = """
def weight_delete(raw, max_cost_ratio=5.0):
    df, report = prepare_trips(raw, max_cost_ratio=max_cost_ratio)
    report["пропусков в весе"] = int(df["weight_t"].isna().sum())
    return df, report
"""
with open("src/prepare.py", "a", encoding="utf-8") as f:
    f.write(weight_t)

import src.prepare
import importlib

# importlib.reload(src.prepare)
from src.prepare import weight_delete

importlib.reload(src.prepare)
cl, t = weight_delete(raw)
print("пропусков в весе:", t["пропусков в весе"])

# 3)
carrier_p = """
def carrier(raw, max_cost_ratio=5.0):
    df, r = prepare_trips(raw, max_cost_ratio=max_cost_ratio)
    carr_nm = set(CANON_CARRIERS.values())
    carr_not = sorted(set(df["carrier"].unique()) - carr_nm)
    r["незнакомые перевозчики"] = carr_not
    uniqie_carr=set(df["carrier"].unique())
    if len(uniqie_carr) != 5:
        print(f'перевозчиков {df["carrier"].unique()}, ожидалось 5')
        print(f'незнакомые перевозчики: {carr_not}')
    print('перевозчиков 5')
    return df, r

    
"""
with open("src/prepare.py", "a", encoding="utf-8") as f:
    f.write(carrier_p)

import src.prepare
import importlib
importlib.reload(src.prepare)
from src.prepare import carrier

raw_p = raw.copy()
f, k = carrier(raw_p)
print(k)

пропусков в весе: 0
перевозчиков 5
{'строк получено': 504, 'удалено дублей': 4, 'дат не распознано': 0, 'перевозчиков после нормализации': 5, 'удалено: нулевой вес': 10, 'удалено: стоимость не бьётся с тарифом': 13, 'строк осталось': 477, 'доля потерь, %': 5.4, 'незнакомые перевозчики': []}


---
# Часть 5. Слой 3: расчёт метрик

Здесь живёт вся аналитика: KPI из практики 10, агрегаты из практики 9, проверка
гипотез из практики 12. Этот слой **не читает файлы и не рисует графики** —
он получает готовую таблицу и возвращает готовые числа.

In [14]:
analyze_code = """
# -*- coding: utf-8 -*-
'''Слой 3: расчёт метрик и проверка гипотез. Файлы не читает, графики не рисует.'''
import pandas as pd
from scipy import stats


def overall_kpi(df):
    '''Сводные показатели по всей компании.'''
    return {
        "рейсов": len(df),
        "выручка": int(df["cost_rub"].sum()),
        "средний чек": round(df["cost_rub"].mean()),
        "медианный чек": round(df["cost_rub"].median()),
        "выручка на км": round(df["cost_rub"].sum() / df["distance_km"].sum(), 1),
        "OTD, %": round((~df["is_delayed"]).mean() * 100, 1),
    }


def kpi_by(df, column):
    '''KPI в разрезе любого признака: перевозчик, месяц, маршрут.'''
    result = (df.groupby(column)
              .agg(trips=("trip_id", "count"),
                   revenue=("cost_rub", "sum"),
                   avg_ticket=("cost_rub", "mean"),
                   otd=("is_delayed", lambda s: (~s).mean() * 100))
              .round(1)
              .sort_values("revenue", ascending=False))
    return result.reset_index()


def compare_otd(df, a, b, alpha=0.05):
    '''Сравнивает доли задержек двух перевозчиков критерием хи-квадрат.'''
    sub = df[df["carrier"].isin([a, b])]
    table = pd.crosstab(sub["carrier"], sub["is_delayed"])
    chi2, p, _, _ = stats.chi2_contingency(table)
    return {"группа A": a, "группа B": b, "chi2": round(chi2, 3),
            "p": round(p, 5), "значимо": bool(p < alpha)}


def worst_and_best_carrier(df, alpha=0.05):
    '''Перевозчики с худшим и лучшим OTD плюс проверка значимости разницы.'''
    otd = df.groupby("carrier")["is_delayed"].mean().sort_values()
    return compare_otd(df, otd.index[-1], otd.index[0], alpha=alpha)
"""

Path("src/analyze.py").write_text(analyze_code.lstrip(), encoding="utf-8")
print("создан src/analyze.py")

создан src/analyze.py


In [15]:
from src.analyze import overall_kpi, kpi_by, worst_and_best_carrier

kpi = overall_kpi(clean)
for k, v in kpi.items():
    print(f"  {k:<16} {v}")
print()
print(kpi_by(clean, "carrier").to_string(index=False))
print()
print("сравнение худшего и лучшего по OTD:")
print(" ", worst_and_best_carrier(clean))

  рейсов           477
  выручка          21651154
  средний чек      45390
  медианный чек    40731
  выручка на км    86.1
  OTD, %           68.8

      carrier  trips   revenue  avg_ticket  otd
АвтоДорСервис     99 4775934.9     48241.8 71.7
     РусКарго    100 4552555.8     45525.6 64.0
   ВолгаТранс     97 4160635.8     42893.2 66.0
ТрансЛогистик     93 4093405.1     44015.1 76.3
      СибПуть     88 4068622.5     46234.3 65.9

сравнение худшего и лучшего по OTD:
  {'группа A': 'РусКарго', 'группа B': 'ТрансЛогистик', 'chi2': np.float64(2.93), 'p': np.float64(0.08692), 'значимо': False}


### Задание 3

1. Вызовите `kpi_by` для столбцов `month` и `route`. Функция одна, разрезов три —
   в этом и смысл параметризации.
2. Добавьте в `analyze.py` функцию `cargo_delay_test(df)`, которая проверяет связь
   типа груза и задержек (хи-квадрат) и возвращает словарь с `chi2`, `p` и выводом.
3. Добавьте в `overall_kpi` показатель «доля рейсов дороже 100 тыс. руб.».

In [16]:
# 1)
from src.analyze import kpi_by, overall_kpi

clean, quality = prepare_trips(raw)

for col in ["carrier", "month", "route"]:
    print(f"{col}")
    print(kpi_by(clean, col).head(3).to_string(index=False))
    print()

carrier
      carrier  trips   revenue  avg_ticket  otd
АвтоДорСервис     99 4775934.9     48241.8 71.7
     РусКарго    100 4552555.8     45525.6 64.0
   ВолгаТранс     97 4160635.8     42893.2 66.0

month
  month  trips    revenue  avg_ticket  otd
2025-04    231 10860834.0     47016.6 71.0
2025-05    246 10790320.2     43863.1 66.7

route
                 route  trips   revenue  avg_ticket  otd
         Москва-Казань     76 5205447.9     68492.7 68.4
      Новосибирск-Омск     54 3285982.4     60851.5 70.4
Москва-Санкт-Петербург     52 3084223.4     59312.0 63.5



In [17]:
#2)
analyze = '''
def cargo_delay_test(df, alpha=0.05):
    table = pd.crosstab(df["cargo_type"], df["is_delayed"])
    chi2, p, dof, _ = stats.chi2_contingency(table)
    return {"chi2": round(chi2, 3), "p": round(p, 5), "dof": int(dof),
            "вывод": "связь есть" if p < alpha else "связи не найденно"}

            
def expensive_share(df, threshold=100_000):
    return round((df["cost_rub"] > threshold).mean() * 100, 1)'''

with open("src/analyze.py", "a", encoding="utf-8") as f:
    f.write(analyze)

import src.analyze
import importlib
importlib.reload(src.analyze)

print("связь типа груза и задержек:", src.analyze.cargo_delay_test(clean))
print("доля рейсов дороже 100 тыс.:", src.analyze.expensive_share(clean), "%")

связь типа груза и задержек: {'chi2': np.float64(3.922), 'p': np.float64(0.56065), 'dof': 5, 'вывод': 'связи не найденно'}
доля рейсов дороже 100 тыс.: 5.0 %


---
# Часть 6. Слой 4: визуализация

Графики строятся **по уже посчитанным метрикам**, а не по сырым данным. Каждая функция
сохраняет файл и возвращает путь к нему — чтобы отчёт знал, что вставлять.

In [18]:
visualize_code = """
# -*- coding: utf-8 -*-
'''Слой 4: построение графиков по готовым метрикам.'''
from pathlib import Path

import matplotlib
matplotlib.use("Agg")      # рисуем в файл, без окна: нужно для запуска из терминала
import matplotlib.pyplot as plt


def plot_revenue_by_month(metrics, out_dir):
    '''Динамика выручки по месяцам.'''
    data = metrics.sort_values("month")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(data["month"], data["revenue"] / 1_000_000, marker="o", linewidth=2.5)
    ax.set_title("Выручка по месяцам")
    ax.set_xlabel("Месяц")
    ax.set_ylabel("Выручка, млн руб.")
    ax.set_ylim(0, (data["revenue"].max() / 1_000_000) * 1.2)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    path = Path(out_dir) / "revenue_by_month.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.close(fig)
    return path


def plot_otd_by_carrier(metrics, out_dir, target=90):
    '''OTD по перевозчикам с линией целевого значения.'''
    data = metrics.sort_values("otd")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    colors = ["#C44E52" if v < target else "#55A868" for v in data["otd"]]
    ax.barh(data["carrier"], data["otd"], color=colors)
    ax.axvline(target, color="black", linestyle="--", linewidth=1.5,
               label=f"цель: {target}%")
    ax.set_title("Доля рейсов без задержки (OTD)")
    ax.set_xlabel("OTD, %")
    ax.set_xlim(0, 100)
    ax.legend()
    fig.tight_layout()

    path = Path(out_dir) / "otd_by_carrier.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.close(fig)
    return path
"""

Path("src/visualize.py").write_text(visualize_code.lstrip(), encoding="utf-8")
print("создан src/visualize.py")

создан src/visualize.py


Обратите внимание на строку `matplotlib.use("Agg")`. Она переключает matplotlib
в режим «рисовать только в файл». Без неё скрипт, запущенный из терминала,
попытается открыть окно и может зависнуть на сервере, где окон нет вообще.

---
# Часть 7. Параметризация

Всё, что может измениться, выносится в **один файл с параметрами**. Правило простое:
если значение встречается в коде больше одного раза или его может захотеться поменять —
оно должно быть в конфиге.

Что обычно выносят: пути, период анализа, пороги, целевые значения KPI, `random_state`.

In [19]:
config_code = """
# -*- coding: utf-8 -*-
'''Все параметры анализа в одном месте.'''

# пути
RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
REPORTS_DIR = "reports"
FIGURES_DIR = "reports/figures"

# какие файлы брать
TRIPS_PATTERN = "trips_*.csv"

# параметры очистки
MAX_COST_RATIO = 5.0     # во сколько раз расчётная стоимость может превышать фактическую

# параметры анализа
OTD_TARGET = 90          # целевая доля рейсов без задержки, %
ALPHA = 0.05             # порог значимости при проверке гипотез
"""

Path("config.py").write_text(config_code.lstrip(), encoding="utf-8")
print("создан config.py")

создан config.py


---
# Часть 8. Сборка: точка входа

`run.py` — единственный файл, который знает о порядке шагов. Он ничего не вычисляет
сам: только вызывает функции слоёв и сохраняет артефакты.

In [20]:
run_code = """
# -*- coding: utf-8 -*-
'''Точка входа: запускает весь анализ от сырых данных до отчёта.'''
import json
from datetime import datetime
from pathlib import Path

import config
from src.load import load_trips, load_carriers
from src.prepare import prepare_trips
from src.analyze import overall_kpi, kpi_by, worst_and_best_carrier
from src.visualize import plot_revenue_by_month, plot_otd_by_carrier


def df_to_markdown(df):
    '''Превращает таблицу в markdown без внешних зависимостей.'''
    header = "| " + " | ".join(str(c) for c in df.columns) + " |"
    sep = "|" + "---|" * len(df.columns)
    rows = ["| " + " | ".join(str(v) for v in row) + " |"
            for row in df.itertuples(index=False)]
    return chr(10).join([header, sep] + rows)


def build_report(kpi, quality, by_carrier, test, figures):
    '''Собирает текстовый отчёт в формате Markdown.'''
    lines = ["# Отчёт по перевозкам", ""]
    lines.append("Сформирован: " + datetime.now().strftime("%d.%m.%Y %H:%M"))

    lines += ["", "## Ключевые показатели", ""]
    for k, v in kpi.items():
        lines.append(f"- **{k}**: {v}")

    lines += ["", "## Качество данных", ""]
    for k, v in quality.items():
        lines.append(f"- {k}: {v}")

    lines += ["", "## По перевозчикам", "", df_to_markdown(by_carrier)]

    verdict = "различие значимо" if test["значимо"] else "различий не обнаружено"
    lines += ["", "## Проверка гипотезы", "",
              f"Сравнение OTD: {test['группа A']} против {test['группа B']} — "
              f"{verdict} (p = {test['p']})."]

    lines += ["", "## Графики", ""]
    for path in figures:
        lines.append(f"![{Path(path).stem}]({Path(path).as_posix()})")

    return chr(10).join(lines)


def main():
    started = datetime.now()
    print("=" * 56)
    print("ЗАПУСК PIPELINE", started.strftime("%Y-%m-%d %H:%M:%S"))
    print("=" * 56)

    for folder in (config.PROCESSED_DIR, config.FIGURES_DIR):
        Path(folder).mkdir(parents=True, exist_ok=True)

    print("[1/5] загрузка")
    raw = load_trips(config.RAW_DIR, config.TRIPS_PATTERN)
    carriers = load_carriers(config.RAW_DIR)

    print("[2/5] подготовка")
    clean, quality = prepare_trips(raw, max_cost_ratio=config.MAX_COST_RATIO)
    clean_path = Path(config.PROCESSED_DIR) / "trips_clean.csv"
    clean.to_csv(clean_path, index=False, encoding="utf-8")
    print(f"    сохранено: {clean_path} ({len(clean)} строк)")

    print("[3/5] расчёт метрик")
    kpi = overall_kpi(clean)
    by_carrier = kpi_by(clean, "carrier")
    by_month = kpi_by(clean, "month")
    test = worst_and_best_carrier(clean, alpha=config.ALPHA)
    by_carrier.to_csv(Path(config.PROCESSED_DIR) / "kpi_by_carrier.csv",
                      index=False, encoding="utf-8")
    by_month.to_csv(Path(config.PROCESSED_DIR) / "kpi_by_month.csv",
                    index=False, encoding="utf-8")

    print("[4/5] графики")
    fig1 = plot_revenue_by_month(by_month, config.FIGURES_DIR)
    fig2 = plot_otd_by_carrier(by_carrier, config.FIGURES_DIR, target=config.OTD_TARGET)
    print(f"    {fig1}")
    print(f"    {fig2}")

    print("[5/5] отчёт")
    report_path = Path(config.REPORTS_DIR) / "report.md"
    report_path.write_text(build_report(kpi, quality, by_carrier, test, [fig1, fig2]),
                           encoding="utf-8")

    manifest = {
        "запуск": started.isoformat(timespec="seconds"),
        "длительность, с": round((datetime.now() - started).total_seconds(), 2),
        "параметры": {"MAX_COST_RATIO": config.MAX_COST_RATIO,
                      "OTD_TARGET": config.OTD_TARGET,
                      "TRIPS_PATTERN": config.TRIPS_PATTERN},
        "качество данных": quality,
        "итоговые метрики": kpi,
    }
    manifest_path = Path(config.PROCESSED_DIR) / "run_manifest.json"
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2),
                             encoding="utf-8")

    print(f"    {report_path}")
    print(f"    {manifest_path}")
    print("ГОТОВО")
    return manifest


if __name__ == "__main__":
    main()
"""

Path("run.py").write_text(run_code.lstrip(), encoding="utf-8")
print("создан run.py")

создан run.py


Теперь весь анализ запускается одной командой. Запустим его прямо из ноутбука
через `subprocess` — ровно так же, как это сделал бы терминал.

In [21]:
import subprocess
import sys

result = subprocess.run([sys.executable, "run.py"], capture_output=True, text=True,
                        encoding="utf-8")
print(result.stdout)
if result.returncode != 0:
    print("ОШИБКА:")
    print(result.stderr[-2000:])

ЗАПУСК PIPELINE 2026-11-25 20:43:43
[1/5] загрузка
    trips_2025_04.csv: 242 строк
    trips_2025_05.csv: 262 строк
[2/5] подготовка
    сохранено: data\processed\trips_clean.csv (477 строк)
[3/5] расчёт метрик
[4/5] графики
    reports\figures\revenue_by_month.png
    reports\figures\otd_by_carrier.png
[5/5] отчёт
    reports\report.md
    data\processed\run_manifest.json
ГОТОВО



In [22]:
# Посмотрим, что получилось
print("АРТЕФАКТЫ:")
for p in sorted(Path("data/processed").glob("*")) + sorted(Path("reports").glob("**/*")):
    if p.is_file():
        print(f"  {p}  ({p.stat().st_size // 1024} КБ)")

print()
print("=" * 56)
print(Path("reports/report.md").read_text(encoding="utf-8")[:900])

АРТЕФАКТЫ:
  data\processed\kpi_by_carrier.csv  (0 КБ)
  data\processed\kpi_by_month.csv  (0 КБ)
  data\processed\run_manifest.json  (0 КБ)
  data\processed\trips_clean.csv  (71 КБ)
  reports\figures\otd_by_carrier.png  (27 КБ)
  reports\figures\revenue_by_month.png  (19 КБ)
  reports\report.md  (1 КБ)

# Отчёт по перевозкам

Сформирован: 25.11.2026 20:43

## Ключевые показатели

- **рейсов**: 477
- **выручка**: 21651154
- **средний чек**: 45390
- **медианный чек**: 40731
- **выручка на км**: 86.1
- **OTD, %**: 68.8

## Качество данных

- строк получено: 504
- удалено дублей: 4
- дат не распознано: 0
- перевозчиков после нормализации: 5
- удалено: нулевой вес: 10
- удалено: стоимость не бьётся с тарифом: 13
- строк осталось: 477
- доля потерь, %: 5.4

## По перевозчикам

| carrier | trips | revenue | avg_ticket | otd |
|---|---|---|---|---|
| АвтоДорСервис | 99 | 4775934.9 | 48241.8 | 71.7 |
| РусКарго | 100 | 4552555.8 | 45525.6 | 64.0 |
| ВолгаТранс | 97 | 4160635.8 | 42893.2 | 66.0 

---
# Часть 9. Проверка воспроизводимости

Главный тест pipeline: **удалить все результаты и получить их заново одной командой**.
Если после этого числа совпали — анализ воспроизводим.

In [23]:
import json
import shutil

# запоминаем результат первого запуска
first = json.loads(Path("data/processed/run_manifest.json").read_text(encoding="utf-8"))

# удаляем ВСЁ, кроме сырых данных и кода
#shutil.rmtree("data/processed")
#shutil.rmtree("reports")
print("удалены data/processed и reports")
print("осталось в data/:", [p.name for p in Path("data").iterdir()])
print()

# запускаем заново
result = subprocess.run([sys.executable, "run.py"], capture_output=True, text=True,
                        encoding="utf-8")
print("код возврата:", result.returncode)

second = json.loads(Path("data/processed/run_manifest.json").read_text(encoding="utf-8"))

print()
print("метрики первого запуска:", first["итоговые метрики"])
print("метрики второго запуска:", second["итоговые метрики"])
print()
print("РЕЗУЛЬТАТЫ СОВПАДАЮТ:" if first["итоговые метрики"] == second["итоговые метрики"]
      else "РЕЗУЛЬТАТЫ РАЗОШЛИСЬ:",
      first["итоговые метрики"] == second["итоговые метрики"])

удалены data/processed и reports
осталось в data/: ['processed', 'raw', 'trips_stat.csv']

код возврата: 0

метрики первого запуска: {'рейсов': 477, 'выручка': 21651154, 'средний чек': 45390, 'медианный чек': 40731, 'выручка на км': 86.1, 'OTD, %': 68.8}
метрики второго запуска: {'рейсов': 477, 'выручка': 21651154, 'средний чек': 45390, 'медианный чек': 40731, 'выручка на км': 86.1, 'OTD, %': 68.8}

РЕЗУЛЬТАТЫ СОВПАДАЮТ: True


### Задание 4

1. Откройте `config.py`, поменяйте `OTD_TARGET` с 90 на 75 и запустите pipeline заново.
   Что изменилось в графике и в отчёте? Сколько строк кода вы для этого поправили?
2. Поменяйте `TRIPS_PATTERN` так, чтобы анализ считался **только по маю**.
   Проверьте, что число рейсов в отчёте изменилось.
3. Верните параметры обратно и убедитесь, что числа вернулись к исходным.
4. Объясните в двух предложениях: почему менять параметры в `config.py` безопаснее,
   чем править их прямо в коде модулей?

In [24]:
# подсказка: config.py можно править и из кода
text = Path("config.py").read_text(encoding="utf-8")
text_75 = text.replace("OTD_TARGET = 90", "OTD_TARGET = 75")
Path("config.py").write_text(text, encoding="utf-8")
result = subprocess.run([sys.executable, "run.py"], capture_output=True, text=True, encoding="utf-8")
print(result.stdout)

ЗАПУСК PIPELINE 2026-11-25 20:43:57
[1/5] загрузка
    trips_2025_04.csv: 242 строк
    trips_2025_05.csv: 262 строк
[2/5] подготовка
    сохранено: data\processed\trips_clean.csv (477 строк)
[3/5] расчёт метрик
[4/5] графики
    reports\figures\revenue_by_month.png
    reports\figures\otd_by_carrier.png
[5/5] отчёт
    reports\report.md
    data\processed\run_manifest.json
ГОТОВО



1) метрики не меняются. Меняется только линия цели на графике и подпись в легенде.

In [25]:
# 2)
text_may = text_75.replace('TRIPS_PATTERN = "trips_*.csv"', 'TRIPS_PATTERN = "trips_2025_05*.csv"')
Path("config.py").write_text(text_may, encoding="utf-8")
rmay = subprocess.run([sys.executable, "run.py"], capture_output=True, text=True, encoding="utf-8")
may = json.loads(Path("data/processed/run_manifest.json").read_text(encoding="utf-8"))
print("число рейсов только за май: ", may["итоговые метрики"]["рейсов"])

число рейсов только за май:  246


In [26]:
# 3)
Path("config.py").write_text(text, encoding="utf-8")
rrestore = subprocess.run([sys.executable, "run.py"], capture_output=True, text=True, encoding="utf-8")
restored = json.loads(Path("data/processed/run_manifest.json").read_text(encoding="utf-8"))
print("число рейсов: ", restored["итоговые метрики"]["рейсов"])

число рейсов:  477


3) После возврата параметров результат совпадает с исходным
4) параметр в config лежит в одном месте и виден целиком; параметр внутри модуля приходится искать, он может встречаться несколько раз, и легко поправить не во всех местах.

---
# Часть 10. Артефакты и манифест

**Артефакт** — любой файл, который создаёт pipeline. Их три вида, и относиться
к ним надо по-разному:

| Вид | Примеры | Коммитить в git? | Можно удалять? |
|---|---|---|---|
| исходные данные | `data/raw/*` | да, если небольшие | **нет**, восстановить неоткуда |
| промежуточные | `data/processed/*.csv` | нет | да, пересоздаются |
| итоговые | `reports/report.md`, графики | по договорённости | да, пересоздаются |

Отсюда `.gitignore` для такого проекта:

```
.venv/
__pycache__/
data/processed/
reports/figures/
```

**Манифест запуска** — отдельный артефакт, который отвечает на вопрос
«откуда взялись эти числа». В нём: когда запускали, с какими параметрами,
что получилось. Без него через месяц невозможно понять, почему в двух отчётах
разные цифры.

In [27]:
manifest = json.loads(Path("data/processed/run_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(manifest, ensure_ascii=False, indent=2))

{
  "запуск": "2026-11-25T20:44:04",
  "длительность, с": 0.2,
  "параметры": {
    "MAX_COST_RATIO": 5.0,
    "OTD_TARGET": 90,
    "TRIPS_PATTERN": "trips_*.csv"
  },
  "качество данных": {
    "строк получено": 504,
    "удалено дублей": 4,
    "дат не распознано": 0,
    "перевозчиков после нормализации": 5,
    "удалено: нулевой вес": 10,
    "удалено: стоимость не бьётся с тарифом": 13,
    "строк осталось": 477,
    "доля потерь, %": 5.4
  },
  "итоговые метрики": {
    "рейсов": 477,
    "выручка": 21651154,
    "средний чек": 45390,
    "медианный чек": 40731,
    "выручка на км": 86.1,
    "OTD, %": 68.8
  }
}


### Задание 5

1. Добавьте в манифест версии библиотек: `pandas.__version__` и `sys.version`.
   Зачем это нужно знать через год?
2. Добавьте в манифест список обработанных файлов с их размерами.
3. Напишите `.gitignore` для проекта и объясните каждую строку.
4. **Проверка на коллегу.** Представьте, что отдаёте проект другому человеку.
   Что ему нужно сделать, чтобы получить ваш отчёт? Напишите инструкцию
   из трёх команд — это и есть содержимое будущего `README.md`.

In [28]:
# 1) - 2): доработайте манифест в run.py и перезапустите
import platform

import pandas as pd

manifest = json.loads(Path("data/processed/run_manifest.json").read_text(encoding="utf-8"))

manifest["окружение"] = {
    "python": sys.version.split()[0],
    "pandas": pd.__version__,
    "платформа": platform.system(),
}
manifest["исходные файлы"] = [
    {"файл": p.name, "размер, КБ": round(p.stat().st_size / 1024, 1)}
    for p in sorted(Path("data/raw").glob("*.csv"))
]

print(json.dumps({"окружение": manifest["окружение"],
                  "исходные файлы": manifest["исходные файлы"]},
                 ensure_ascii=False, indent=2))

{
  "окружение": {
    "python": "3.11.16",
    "pandas": "3.0.5",
    "платформа": "Windows"
  },
  "исходные файлы": [
    {
      "файл": "carriers.csv",
      "размер, КБ": 0.2
    },
    {
      "файл": "trips_2025_04.csv",
      "размер, КБ": 18.5
    },
    {
      "файл": "trips_2025_05.csv",
      "размер, КБ": 20.2
    }
  ]
}


 ```
 3) 
 .venv/              окружение: у каждого своё, восстанавливается из requirements.txt
 __pycache__/        кэш Python, создаётся автоматически
 data/processed/     результат работы pipeline, пересоздаётся командой run.py
 reports/figures/    графики, пересоздаются
 .env                пароли и ключи (нельзя добавлять в репозиторий)
```

> **README.md проекта (задание 5, пункт 4):**
>
> ```
> 1. python -m venv .venv  и активировать окружение
> 2. pip install -r requirements.txt
> 3. python run.py
> ```

---
## Итоговое задание: перенесите свой анализ

Возьмите любую свою работу с курса — практику 9, 10 или 11 — и переоформите её
в виде pipeline по сегодняшней структуре. Требования минимальные:

- три слоя в `src/` (загрузка, подготовка, расчёт);
- параметры в `config.py`;
- `run.py`, выполняющий всё целиком;
- хотя бы один сохраняемый артефакт и отчёт;
- проект запускается командой `python run.py` в чистой папке после `pip install -r requirements.txt`.

Это и есть то, как выглядит аналитическая работа, которую можно показать
работодателю: не ноутбук с шестьюдесятью ячейками, а проект, который запускается.

---
## Шпаргалка по структуре

```
project/
├── README.md              как запустить
├── requirements.txt       зависимости
├── config.py              все параметры
├── run.py                 точка входа
├── src/
│   ├── load.py            читает
│   ├── prepare.py         чистит
│   ├── analyze.py         считает
│   └── visualize.py       рисует
├── data/
│   ├── raw/               только чтение!
│   └── processed/         пересоздаётся
└── reports/               пересоздаётся
```

**Правила:**

1. `data/raw/` никогда не изменяется.
2. Каждый слой делает одно дело и не лезет в соседний.
3. Параметры — в конфиге, не в коде.
4. Всё, что можно пересоздать, не коммитится в git.
5. Каждый запуск оставляет манифест: когда, с какими параметрами, что получилось.

## Отправляем работу

```
# терминал
git add .
git commit -m "Практика 13: воспроизводимый аналитический pipeline"
git push
```

## Чек-лист сдачи

- [ ] Задание 1: загрузка по шаблону, обработка отсутствующей папки, функция `load_all`
- [ ] Задание 2: другой порог очистки, проверка на незнакомых перевозчиков
- [ ] Задание 3: `kpi_by` для трёх разрезов, функция проверки гипотезы
- [ ] Задание 4: параметры меняются только в `config.py`
- [ ] Задание 5: манифест дополнен, написан `.gitignore` и инструкция запуска
- [ ] `python run.py` отрабатывает от начала до конца и создаёт отчёт
- [ ] После удаления `data/processed` и `reports` всё восстанавливается одной командой
- [ ] Всё запушено в репозиторий

## Что унести с пары — и со всего курса

Воспроизводимость — это не аккуратность ради аккуратности. Это ответ на вопрос,
который вам зададут в первый же рабочий месяц: **«откуда эта цифра?»**

Если анализ лежит в ноутбуке, ответ занимает час и часто заканчивается словами
«кажется, я что-то поменял». Если анализ — это pipeline, ответ занимает минуту:
вот сырые данные, вот код, вот параметры, вот манифест запуска. Запустите сами
и получите то же самое.

За курс мы прошли весь путь: окружение и git → базовый Python → pandas → загрузка
из разных источников → очистка → агрегация → статистика → визуализация → проверка
гипотез. Сегодня эти шаги впервые собрались в то, чем они являются в работе, —
**в один воспроизводимый процесс**.